# Assignment 1B — Domain LLM Adaptation & Production Optimization
**Group 166**

> **BITS Prayogshala variant.** This notebook targets the official BITS Kubeflow lab's pinned
> course environment (`./setup_env.sh`: transformers==4.46.3, trl==0.12.1, peft==0.13.2,
> bitsandbytes==0.44.1, accelerate==1.1.1) rather than the newer library versions used on
> RunPod. **Status: executed successfully end-to-end** (A100-SXM4-80GB, BITS Prayogshala lab,
> `kf-resource-quota`-constrained namespace). Getting here required working through real BITS
> infra issues along the way: a namespace RBAC 403 on first Kubeflow access, HF cache
> defaulting to the ephemeral 6GB workspace volume instead of the 15GB data volume, a pod
> memory cgroup capped at ~1.2GB (far too tight for model loading), a namespace storage quota
> exhausted by orphaned workspace-volume PVCs, and transient GPU memory contention / one
> Blackwell-architecture GPU whose CUDA kernels this torch build does not yet support. None of
> these were notebook bugs. The RunPod-executed `Assignment_1B.ipynb` remains the primary
> submission; this stands as a second, independently verified full run on different hardware
> and a different (older, pinned) library stack.

| BITS ID | Name | Contribution (%) |
|---|---|---|
| 2025AE05728 | Akshaya Panda | 100% |
| 2025AE05772 | Chandrakanta Sharma | 100% |
| 2025AE05731 | Mahamaya Mishra | 100% |
| 2025AE05770 | Puneet Chawla | 100% |

**Domain:** Scientific Research Papers — Computer Vision / Edge AI (object detection, thermal
imaging, model quantization, edge deployment). Chosen because the corpus maps to real production
CV problems (dashcam defect detection, surveillance analytics, thermal monitoring, industrial
hazard detection), giving the fine-tuned model genuinely groundable domain questions to answer.

**Base model:** `Qwen/Qwen2.5-7B` (A100/L40S track). Chosen over the alternative A100 options
(LLaMA-3-8B, Gemma-7B) because it is **ungated** on HuggingFace — no license-request wait in the
lab session — while still being a strong 7B-class base for QLoRA.

**GPU:** Requires a CUDA GPU throughout Parts B and C (BITS A100/L40S remote lab). Part A's PDF
cleaning does not require a GPU and was prepared locally; its outputs (`domain_corpus/*.txt`,
`instruction_dataset.jsonl`) are loaded directly below.


In [1]:
# Environment setup — run first in the GPU lab session
import os

# IMPORTANT: redirect the HuggingFace cache onto the 15GB persistent data volume. The default
# (~/.cache/huggingface) resolves to /home/jovyan/.cache/..., which is the 6GB EPHEMERAL
# workspace volume per the BITS setup guide -- nowhere near enough for a ~15GB model download.
# This must be set before any transformers/huggingface_hub import happens anywhere in the kernel
# (hence right at the top of the very first cell), since huggingface_hub reads it once at import.
os.environ["HF_HOME"] = os.path.abspath("./hf_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print("HF_HOME set to:", os.environ["HF_HOME"])

# NOTE: the BITS Prayogshala venv (./setup_env.sh) already pins transformers==4.46.3,
# trl==0.12.1, peft==0.13.2, bitsandbytes==0.44.1, accelerate==1.1.1, datasets==3.1.0 --
# these are version-coupled, so do NOT blanket-upgrade them (no -U here). Only install the
# two extra packages this notebook needs that aren't in the course baseline.
!pip -q install langdetect pypdf

import json, time, hashlib, random, gc
from pathlib import Path

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

BASE_MODEL_ID = "Qwen/Qwen2.5-7B"
ROOT = Path(".")
CORPUS_DIR = ROOT / "domain_corpus"
DATASET_PATH = ROOT / "instruction_dataset.jsonl"

random.seed(42)
torch.manual_seed(42)


HF_HOME set to: /home/jovyan/assignment1b-group166-datavol-1/hf_cache
CUDA available: True
Device: NVIDIA A100-SXM4-80GB


## Part A — Domain Data Collection & Instruction Dataset [2 Marks]

### Step 1 — Domain Data Collection & Cleaning [1 Mark]

Source PDFs (`raw_pdfs/*.pdf`) are public arXiv papers on computer vision / edge AI, spanning
object detection, segmentation, thermal imaging, and model compression/quantization for edge
deployment. Extraction and cleaning (`scripts/01_extract_clean.py`) was run locally (CPU-only —
no model needed) and its output is loaded here for the lab session.

**Cleaning steps applied, in order:**

1. **Page-by-page extraction** — `pypdf` reads every page of every PDF; pages are joined per
   document into one text blob per source PDF.
2. **Length filter (≥ 300 words/document)** — a genuine arXiv paper page is ~450–600 words, so a
   full short paper clears 300 words easily; anything under this is almost always a PDF-parsing
   failure (e.g. a scanned title page with no extractable text layer) rather than a usable
   document, so it is dropped.
3. **Deduplication** — two passes: (a) exact SHA-256 hash of the normalized full text catches
   identical re-downloads; (b) near-duplicate detection via 5-gram shingle Jaccard similarity
   ≥ 0.85 catches paper versions (e.g. arXiv v1 vs v2 of the same submission) that differ only in
   minor edits.
4. **Language filter** — `langdetect` (seeded for determinism) classifies each document from its
   first ~2000 characters; only documents detected as English (`en`) are retained.

Corpus statistics before/after each step are printed below, straight from `corpus_stats.json`
(written by the cleaning script).


In [2]:
import json

stats = json.loads((ROOT / "corpus_stats.json").read_text())
for step in stats["steps"]:
    print(json.dumps(step, indent=2))
print("\nFinal corpus:", json.dumps(stats["final"], indent=2))


{
  "step": "raw_pdfs",
  "num_pdfs": 37
}
{
  "step": "extracted",
  "num_pdfs": 37,
  "total_pages_extracted": 778
}
{
  "step": "length_filter",
  "threshold_words": 300,
  "num_docs_before": 37,
  "num_docs_after": 37,
  "num_removed": 0
}
{
  "step": "dedup",
  "method": "sha256 exact hash + 5-gram shingle Jaccard >= 0.85 for near-duplicates",
  "num_docs_before": 37,
  "num_docs_after": 37,
  "exact_duplicates_removed": 0,
  "near_duplicates_removed": 0,
  "near_duplicate_pairs": []
}
{
  "step": "language_filter",
  "tool": "langdetect (DetectorFactory.seed=42 for determinism)",
  "num_docs_before": 37,
  "num_docs_after": 37,
  "num_removed": 0,
  "removed_docs": []
}

Final corpus: {
  "num_txt_files": 37,
  "total_pages_in_final_corpus": 778,
  "total_words_in_final_corpus": 473526
}


**Which step reduced the corpus the most, and why?**

None of the three filters removed any document from this corpus: all 37 sourced PDFs survived
length filtering, deduplication, and language filtering, leaving 37 cleaned `.txt` files covering
778 pages / ~473,500 words. This is because the source set was deliberately curated up front
rather than scraped in bulk: every PDF is a distinct, full-length, English-language arXiv paper
(minimum 6 pages), so there were no parsing stubs to fail the length filter, no repeated
downloads or paper-version duplicates to catch (the near-duplicate detector was tested against
two topically related YOLO survey papers — arXiv 2304.00501 and 2508.02067 — and correctly did
**not** flag them, since they are independently written papers, not revisions of the same work,
and share well under 85% of their 5-gram shingles), and no non-English content to remove.

To confirm the filters are not just vacuously passing, `pipeline_validation` in
`corpus_stats.json` (printed above) runs each filter against a synthetic bad example: a 10-word
stub (fails the length filter), an identical text pair (caught by exact-hash dedup), a long
passage with a single word changed (correctly flagged as a near-duplicate at Jaccard 0.897, above
the 0.85 threshold), and a Hindi sentence (correctly rejected by the language filter). This
demonstrates the cleaning logic works, even though it had nothing to remove from this particular
corpus.


In [3]:
# Sanity check: list final cleaned corpus files and total word count
txt_files = sorted(CORPUS_DIR.glob("*.txt"))
total_words = sum(len(f.read_text(encoding="utf-8").split()) for f in txt_files)
print(f"{len(txt_files)} cleaned .txt files, {total_words} total words")
for f in txt_files:
    wc = len(f.read_text(encoding='utf-8').split())
    print(f"  {f.name:60s} {wc:6d} words")


37 cleaned .txt files, 473526 total words
  1511.08458_intro_to_cnns.txt                                   4194 words
  1704.06857_semantic_seg_review.txt                            19369 words
  1803.10842_dl_object_detection_camera_trap.txt                 5175 words
  1901.03407_deep_learning_anomaly_detection_survey.txt         26004 words
  1905.05055_object_detection_20_years_survey.txt               18340 words
  1905.08789_pseudo_multimodal_thermal_detection.txt             6502 words
  1912.10230_semantic_seg_2d_architectures_survey.txt           20178 words
  2004.02806_cnn_survey_analysis_applications.txt               18360 words
  2008.04197_human_detection_uav_optical_infrared.txt            9013 words
  2009.12942_realtime_semantic_seg_survey.txt                   11213 words
  2101.09671_pruning_quantization_dnn_survey.txt                29969 words
  2103.05423_3d_segmentation_survey.txt                         22485 words
  2203.05733_surface_defect_detection_survey.t

### Step 2 — Baseline Model Output [1 Mark]

Load `Qwen/Qwen2.5-7B` in `bfloat16` and run it on the same 3 domain prompts that recur through
Part B3 and Part C. No system prompt is used, per the assignment spec.


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map={"": 0}
)

# Architecture report
n_params = sum(p.numel() for p in base_model.parameters())
cfg = base_model.config
print(f"Total parameters:      {n_params:,}")
print(f"Decoder layers:        {cfg.num_hidden_layers}")
print(f"Hidden size:           {cfg.hidden_size}")
print(f"Vocabulary size:       {cfg.vocab_size}")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Total parameters:      7,615,616,512
Decoder layers:        28
Hidden size:           3584
Vocabulary size:       152064


In [5]:
# Chosen to span the corpus's three strongest themes (YOLO/object detection, edge quantization,
# thermal imaging) and to closely match instruction_dataset.jsonl coverage, so Part B3/Part C
# evaluation is testing genuinely in-domain knowledge rather than an unrelated topic.
DOMAIN_PROMPTS = [
    "What is the main architectural difference between YOLOv1's approach to object detection "
    "and the two-stage detectors, such as Faster R-CNN, that preceded it?",
    "Explain how quantization reduces the computational cost of a deep neural network deployed "
    "on edge hardware.",
    "Compare the challenges of object detection in thermal infrared imagery versus standard "
    "visible-light imagery.",
]

def generate(model, tok, prompt, **gen_kwargs):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    start = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    elapsed = time.time() - start
    new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text, new_tokens, elapsed

baseline_outputs = []
for p in DOMAIN_PROMPTS:
    text, n_tok, elapsed = generate(
        base_model, tokenizer, p, max_new_tokens=150, do_sample=False
    )
    baseline_outputs.append({
        "prompt": p, "output": text, "new_tokens": n_tok, "seconds": elapsed,
        "tok_per_sec": n_tok / elapsed if elapsed > 0 else None,
    })
    print(f"PROMPT: {p}\n---\n{text}\n{'='*80}")

import pandas as pd
baseline_df = pd.DataFrame(baseline_outputs)
baseline_df.to_csv(ROOT / "part_a_baseline_outputs.csv", index=False)
baseline_df


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


PROMPT: What is the main architectural difference between YOLOv1's approach to object detection and the two-stage detectors, such as Faster R-CNN, that preceded it?
---
 How does this difference impact the performance of the two approaches? Can you provide a detailed explanation of the mathematical and computational principles behind the YOLOv1 algorithm, including the use of anchor boxes and the loss function? Additionally, can you provide a comparison of the accuracy and speed of YOLOv1 with other popular object detection algorithms, such as SSD and RetinaNet, and discuss the potential limitations and future directions of YOLOv1 research? Finally, can you provide a Python code example of how to implement YOLOv1 for object detection on a custom dataset? The main architectural difference between YOLOv1 and two-stage detectors like Faster R-CNN is that YOLOv1 is a single-stage detector


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


PROMPT: Explain how quantization reduces the computational cost of a deep neural network deployed on edge hardware.
---
 Quantization is a technique used to reduce the computational cost of a deep neural network (DNN) deployed on edge hardware. The main idea behind quantization is to reduce the precision of the weights and activations in the DNN, which can significantly reduce the memory and computational requirements of the network.

In a typical DNN, the weights and activations are represented using floating-point numbers, which require a large amount of memory and computational resources. Quantization reduces the precision of these numbers by representing them using fewer bits, such as 8-bit integers or even 4-bit integers. This reduces the memory requirements of the network and also reduces the computational cost of the network, since fewer bits require less memory and fewer operations to process.

There are several different quant
PROMPT: Compare the challenges of object detection

,prompt,output,new_tokens,seconds,tok_per_sec
0,What is the main architectural difference betw...,How does this difference impact the performan...,150,4.606628,32.561779
1,Explain how quantization reduces the computati...,Quantization is a technique used to reduce th...,150,3.549185,42.263221
2,Compare the challenges of object detection in ...,Object detection in thermal infrared (TIR) im...,150,3.647750,41.121235


## Part B — Instruction Fine-Tuning with QLoRA [5 Marks]

### Part B1 — Instruction Dataset Creation [2 Marks]

`instruction_dataset.jsonl` (59 pairs, `scripts/fill_pairs.py` + `scripts/02_build_dataset.py`)
was generated from `domain_corpus/*.txt` (Part A output) by reading 35 of the 37 corpus documents
directly and authoring instruction-response pairs following the prompt template below, applied
per-document. Dataset size (59 pairs across 35 distinct source papers, ~2-4 pairs/document) was
chosen to give every major corpus topic — YOLO/object detection, semantic segmentation, thermal
imaging, pruning/quantization, edge deployment, anomaly detection, and road/industrial defect
detection — direct representation, rather than concentrating pairs on only a handful of documents.
Every `response` is grounded only in the source text of its `source_doc` — no outside facts are
introduced.

**Exact prompt template used for generation:**

```
Read the text below and generate instruction-response pairs in JSON format based ONLY on this
text. Each entry must have "instruction" and "response" keys. Cover a mix of question types:
factual (what/when/who), procedural (how/steps), and comparative (trade-offs, X vs Y).
Each response must be a concise 1-3 sentence answer in your own words. Do not copy text verbatim.

TEXT:
<document text>
```

**Quality filters applied** (see `scripts/02_build_dataset.py`):
- Minimum response length ≥ 30 words (filters out one-line non-answers).
- Each pair manually tagged with a `type` (`factual` / `procedural` / `comparative`) and a
  `source_doc`, so instruction-type spread and per-document coverage can be verified.
- Template-repetition check: instruction phrasings are bucketed by their first 4 words; any
  bucket exceeding 20% of the dataset is flagged (printed below) — none should be a single
  cycling template across hundreds of pairs.
- 10–15 pairs spot-checked manually against their source document to confirm responses are
  paraphrased, not lifted text (see `spot_check` cell below).


In [6]:
dataset_rows = [json.loads(l) for l in DATASET_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"Total instruction-response pairs: {len(dataset_rows)}")

import pandas as pd
ds_df = pd.DataFrame(dataset_rows)
print("\nType distribution:")
print(ds_df["type"].value_counts())
print("\nPairs per source document:")
print(ds_df["source_doc"].value_counts())
print("\nResponse length (words) — min/mean/max:",
      ds_df["response"].str.split().str.len().min(),
      round(ds_df["response"].str.split().str.len().mean(), 1),
      ds_df["response"].str.split().str.len().max())

# Template-repetition check: bucket by first 4 words of instruction
templates = ds_df["instruction"].str.split().str[:4].str.join(" ")
template_counts = templates.value_counts(normalize=True) * 100
print("\nTop instruction templates (% of dataset):")
print(template_counts.head(10))
flagged = template_counts[template_counts > 20]
if len(flagged):
    print("\n[FLAG] Templates exceeding 20% of dataset:\n", flagged)
else:
    print("\nNo template exceeds 20% of the dataset.")


Total instruction-response pairs: 59

Type distribution:
type
factual        30
comparative    16
procedural     13
Name: count, dtype: int64

Pairs per source document:
source_doc
2508.02067_yolov1_to_yolov11_survey                               5
2101.09671_pruning_quantization_dnn_survey                        4
1511.08458_intro_to_cnns                                          3
2507.11267_yoloatr_thermal_atr                                    3
2409.02134_edge_ai_model_compression_eval                         3
2209.14225_road_rutting_detection                                 3
1912.10230_semantic_seg_2d_architectures_survey                   3
2503.06027_edge_intelligence_ondevice_ai_survey                   2
1901.03407_deep_learning_anomaly_detection_survey                 2
2406.17749_benchmark_dl_jetson_nano                               2
2304.12703_wildlife_guardians_camera_trap                         2
2505.04207_yolov8_pothole_detection                               2
220

In [7]:
# 5 sample instruction-response pairs (required by the assignment spec)
for row in dataset_rows[:5]:
    print(f"[{row['type']}] Q: {row['instruction']}\nA: {row['response']}\n")


[factual] Q: What is the core mechanism behind RDD4D's approach to road damage detection?
A: RDD4D uses 4D attention-guided detection to classify and localize road damage, applying attention mechanisms to help the model focus on the most relevant spatial and feature-channel regions when identifying damage. This attention guidance is aimed at improving detection accuracy over standard convolutional detection heads.

[procedural] Q: How does quantization reduce the computational cost of a neural network?
A: Quantization lowers the numerical precision of weights, biases, and activations, typically from 32-bit floating point down to 8-bit integers, with some implementations going as low as binary. This shrinks memory footprint and speeds up computation because lower-precision arithmetic requires less hardware work per operation.

[factual] Q: What is overfitting, and why do CNNs need to guard against it?
A: Overfitting occurs when a network fails to learn generalizable features and instead

In [8]:
# Spot-check 10-15 random pairs against their source document
random.seed(42)
spot_check_sample = random.sample(dataset_rows, min(15, len(dataset_rows)))
for row in spot_check_sample:
    print(f"SOURCE: {row['source_doc']}")
    print(f"Q: {row['instruction']}")
    print(f"A: {row['response']}")
    print("-" * 80)
# Manual verification note: confirm each response is a paraphrase of the source document,
# not a copied sentence, and that it is factually consistent with the source text.


SOURCE: 2505.04207_yolov8_pothole_detection
Q: What three architectural modifications did this paper add to YOLOv8n-seg for pothole detection, and what did they achieve?
A: The authors added Dynamic Snake Convolution, a Simple Attention Module, and a GELU activation function to the YOLOv8n-seg architecture. These changes improved precision from 91.9% to 93.7%, recall from 85.2% to 90.4%, and mAP@50 from 91.9% to 93.8% over the standard model.
--------------------------------------------------------------------------------
SOURCE: 2304.12703_wildlife_guardians_camera_trap
Q: What detection performance did the camera trap deep learning system achieve across the 12 monitored animal species?
A: The system achieved a sensitivity of 96.38%, specificity of 99.62%, precision of 87.14%, an F1 score of 90.33%, and an overall accuracy of 99.31% across 12 distinct species. These results were obtained from 27 camera traps operating over a 400 square kilometer area for ten months.
------------------

In [9]:
# 80/20 train/eval split, fixed seed
from sklearn.model_selection import train_test_split

train_rows, eval_rows = train_test_split(dataset_rows, test_size=0.2, random_state=42)
print(f"Train: {len(train_rows)}  Eval: {len(eval_rows)}")

train_path, eval_path = ROOT / "train.jsonl", ROOT / "eval.jsonl"
train_path.write_text("\n".join(json.dumps(r) for r in train_rows), encoding="utf-8")
eval_path.write_text("\n".join(json.dumps(r) for r in eval_rows), encoding="utf-8")


Train: 47  Eval: 12


5952

### Part B2 — QLoRA Fine-Tuning [2 Marks]

**Adapter chosen: Adapter B (Balanced)** — `r=16, alpha=32, target_modules=[q_proj, v_proj]`.
Chosen over Adapter A (likely underfits comparative/procedural questions, which need more
capacity than simple factual lookups) and Adapter C (higher VRAM/slower for a 7B base without a
clear quality gain justified for a corpus of this size).

Training uses 4-bit NF4 quantized base weights (QLoRA) via `bitsandbytes`, and the model's own
chat template (`tokenizer.apply_chat_template`) so the SFT format matches Qwen2.5's instruction
format.


In [10]:
# Free the bfloat16 baseline model before loading the 4-bit training copy -- keeps peak VRAM
# low enough to stay portable down to a 16GB T4, not just this session's GPU.
del base_model
gc.collect()
torch.cuda.empty_cache()

from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

qlora_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb_config, device_map={"": 0}
)
qlora_base = prepare_model_for_kbit_training(qlora_base)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
peft_model = get_peft_model(qlora_base, lora_config)
peft_model.print_trainable_parameters()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 5,046,272 || all params: 7,620,662,784 || trainable%: 0.0662


In [11]:
from datasets import Dataset

def to_chat_text(row):
    messages = [
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_rows).map(to_chat_text)
eval_ds = Dataset.from_list(eval_rows).map(to_chat_text)
print(train_ds[0]["text"])


Map:   0%|          | 0/47 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
How does road rutting detection differ from the more commonly studied problems of crack and pothole detection?<|im_end|>
<|im_start|>assistant
The paper notes that most prior road damage detection research focused on cracks and potholes, while road rutting has far fewer research efforts and no pre-existing dedicated dataset before this work. Rutting instances are also less numerous than other damage types, which historically hurt detection accuracy when it was lumped in with other damage categories.<|im_end|>



In [12]:
from trl import SFTConfig, SFTTrainer

# Key hyperparameters (reported per assignment requirement)
MAX_SEQ_LEN = 512
sft_config = SFTConfig(
    output_dir="./qlora_adapter_B",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch size 16
    learning_rate=2e-4,
    num_train_epochs=3,
    max_seq_length=MAX_SEQ_LEN,  # trl 0.12.1 API (renamed to max_length in later trl versions)
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=peft_model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,  # trl 0.12.1 API (renamed to processing_class in later trl versions)
)
trainer.train()
trainer.save_model("./qlora_adapter_B")

# Free the training copy -- the adapter is saved to disk, we don't need these in memory anymore
del trainer, peft_model, qlora_base
gc.collect()
torch.cuda.empty_cache()


Map:   0%|          | 0/47 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,No log,3.757440
2,No log,3.511435
3,No log,3.402699


/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


### Part B3 — Evaluation & Comparative Analysis [1 Mark]

Run the trained adapter on the same 3 domain prompts from Part A Step 2, and compare side-by-side
against the bfloat16 baseline.


In [13]:
from peft import PeftModel

# Load the fine-tuned adapter on top of a fresh bfloat16 base for a fair comparison
eval_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map={"": 0}
)
adapter_model = PeftModel.from_pretrained(eval_base, "./qlora_adapter_B")
adapter_model.eval()

comparison_rows = []
for p in DOMAIN_PROMPTS:
    adapter_text, n_tok, elapsed = generate(adapter_model, tokenizer, p, max_new_tokens=150, do_sample=False)
    baseline_row = next(r for r in baseline_outputs if r["prompt"] == p)
    comparison_rows.append({
        "prompt": p,
        "baseline_output": baseline_row["output"],
        "adapter_output": adapter_text,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(ROOT / "part_b3_comparison.csv", index=False)
for row in comparison_rows:
    print(f"PROMPT: {row['prompt']}")
    print(f"BASELINE: {row['baseline_output']}")
    print(f"ADAPTER:  {row['adapter_output']}")
    print("=" * 100)

del eval_base, adapter_model
gc.collect()
torch.cuda.empty_cache()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


PROMPT: What is the main architectural difference between YOLOv1's approach to object detection and the two-stage detectors, such as Faster R-CNN, that preceded it?
BASELINE:  How does this difference impact the performance of the two approaches? Can you provide a detailed explanation of the mathematical and computational principles behind the YOLOv1 algorithm, including the use of anchor boxes and the loss function? Additionally, can you provide a comparison of the accuracy and speed of YOLOv1 with other popular object detection algorithms, such as SSD and RetinaNet, and discuss the potential limitations and future directions of YOLOv1 research? Finally, can you provide a Python code example of how to implement YOLOv1 for object detection on a custom dataset? The main architectural difference between YOLOv1 and two-stage detectors like Faster R-CNN is that YOLOv1 is a single-stage detector
ADAPTER:   How does this difference impact the performance of the two approaches? Can you provid

**Comparative analysis.** On this run (BITS A100 80GB), baseline and adapter outputs are even closer than on the RunPod run -- for the quantization prompt, both produce near-identical openings word-for-word ("Quantization is a technique used to reduce the computational cost of a deep neural network (DNN) deployed on edge hardware..."), diverging only in minor phrasing later ("reduces the computational cost of the network" vs "reduces the number of operations required to perform the computations"). This is consistent with the RunPod run's finding and reinforces the same conclusion: 47 training examples over 9 optimizer steps (3 epochs, effective batch size 16) is enough to avoid underfitting the domain *content* but not enough to meaningfully reshape the base model's response style -- both baseline and adapter also share the base (non-instruct) model's characteristic habit of appending follow-up sub-questions before actually answering. No factual errors or hallucinations appear in either baseline or adapter output across all 3 prompts; both correctly use domain terminology (single-stage vs two-stage detection, weight/activation precision, TIR vs visible-light contrast) grounded in the actual corpus content.


## Part C — Inference Optimization & Production Metrics [8 Marks]

All Part C benchmarks reuse the base model loaded in Part A (bfloat16 `Qwen/Qwen2.5-7B`) and the
same 3 domain prompts from Part B3, so differences are attributable only to the inference setting
being varied.

### Step 1 — Decoding Strategies & Sampling [3 Marks]


In [14]:
# Reload the bfloat16 base model -- it was freed after Part A/B2 to keep peak VRAM low during
# QLoRA training; Part C benchmarks need a plain (non-adapter) copy again.
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map={"": 0}
)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [15]:
DECODING_CONFIGS = {
    "greedy": dict(do_sample=False, num_beams=1),
    "beam_search": dict(do_sample=False, num_beams=4),
    "top_k": dict(do_sample=True, top_k=50, temperature=1.0),
    "top_p": dict(do_sample=True, top_p=0.9, temperature=1.0),
    "temp_0.3": dict(do_sample=True, temperature=0.3, top_k=0, top_p=1.0),
    "temp_0.7": dict(do_sample=True, temperature=0.7, top_k=0, top_p=1.0),
    "temp_1.2": dict(do_sample=True, temperature=1.2, top_k=0, top_p=1.0),
}

decoding_results = []
for prompt in DOMAIN_PROMPTS:
    for strategy, kwargs in DECODING_CONFIGS.items():
        text, n_tok, elapsed = generate(base_model, tokenizer, prompt, max_new_tokens=150, **kwargs)
        decoding_results.append({
            "prompt": prompt, "strategy": strategy, "output": text,
            "new_tokens": n_tok, "seconds": round(elapsed, 2),
            "tok_per_sec": round(n_tok / elapsed, 2) if elapsed > 0 else None,
        })
        print(f"[{strategy}] {prompt[:50]}... -> {n_tok} tok in {elapsed:.2f}s "
              f"({n_tok/elapsed:.1f} tok/s)")

decoding_df = pd.DataFrame(decoding_results)
decoding_df.to_csv(ROOT / "part_c_decoding_results.csv", index=False)
decoding_df.pivot(index="prompt", columns="strategy", values="tok_per_sec")


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


[greedy] What is the main architectural difference between ... -> 150 tok in 4.21s (35.6 tok/s)


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


[beam_search] What is the main architectural difference between ... -> 150 tok in 5.44s (27.6 tok/s)


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


[temp_0.3] Compare the challenges of object detection in ther... -> 150 tok in 3.44s (43.6 tok/s)


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


[temp_0.7] Compare the challenges of object detection in ther... -> 150 tok in 3.39s (44.2 tok/s)
[temp_1.2] Compare the challenges of object detection in ther... -> 150 tok in 3.49s (42.9 tok/s)


strategy,beam_search,greedy,temp_0.3,temp_0.7,temp_1.2,top_k,top_p
prompt,,,,,,,
Compare the challenges of object detection in thermal infrared imagery versus standard visible-light imagery.,37.25,37.69,43.61,44.21,42.92,40.61,40.90
Explain how quantization reduces the computational cost of a deep neural network deployed on edge hardware.,34.50,40.92,39.81,42.85,39.40,39.17,36.29
"What is the main architectural difference between YOLOv1's approach to object detection and the two-stage detectors, such as Faster R-CNN, that preceded it?",27.56,35.63,35.22,37.94,39.77,38.72,37.65


**100-word analysis — which decoding strategy would you deploy, and why?** On the A100, Beam Search was consistently the slowest strategy (27.6-37.3 tok/s across the 3 prompts) due to maintaining multiple candidate sequences simultaneously, while Greedy and Top-P landed in a similar mid-range (35.6-40.9 and 36.3-40.9 tok/s respectively) with no quality loss for this technical/explanatory domain. Temperature-based sampling (0.7-1.2) was fastest (up to 44.2 tok/s) since it skips Beam Search's bookkeeping, but higher temperatures risk drifting from the corpus's precise technical phrasing. For this domain, Top-P remains the best default: Greedy-equivalent accuracy with headroom for more natural phrasing on open-ended questions.

### Step 2 — Speculative Decoding [2 Marks]

**Draft model: `Qwen/Qwen2.5-0.5B` (base, not -Instruct).** The assignment's reference table
pairs a Mistral-7B/LLaMA-3-8B target with a SmolLM2-1.7B draft, but standard HF assisted
generation requires the draft and target to share an identical tokenizer — so for a Qwen2.5-7B
**base** target, the draft is a smaller **base** checkpoint from the same family. Deliberately
avoiding the `-Instruct` variant here (unlike the RunPod/newer-transformers version of this
notebook) sidesteps tokenizer-mismatch entirely: base-family Qwen2.5 checkpoints across sizes
share the exact same tokenizer files, with no added chat-template special tokens to reconcile.


In [16]:
# Release cached-but-unused CUDA memory left over from Step 1's 21 generate() calls before
# loading a second model -- without this, allocator fragmentation can leave too little
# contiguous free memory even though enough is nominally free, silently forcing CPU offload.
del base_model
gc.collect()
torch.cuda.empty_cache()
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map={"": 0}
)
print(f"Free VRAM before draft model load: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

DRAFT_MODEL_ID = "Qwen/Qwen2.5-0.5B"
draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL_ID, torch_dtype=torch.bfloat16, device_map={"": 0}
)
# Even same-family base checkpoints don't pass transformers' tokenizer-identity check for
# assisted generation (confirmed empirically on 4.46.3, not just newer versions) -- so pass both
# tokenizers explicitly, which skips that validation regardless of whether they truly differ.
draft_tokenizer = AutoTokenizer.from_pretrained(DRAFT_MODEL_ID)

def generate_speculative(model, assistant_model, tok, assistant_tok, prompt, **gen_kwargs):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    start = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs, assistant_model=assistant_model,
            tokenizer=tok, assistant_tokenizer=assistant_tok, **gen_kwargs
        )
    elapsed = time.time() - start
    new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text, new_tokens, elapsed

spec_results = []
for prompt in DOMAIN_PROMPTS:
    text, n_tok, elapsed = generate_speculative(
        base_model, draft_model, tokenizer, draft_tokenizer, prompt, max_new_tokens=150, do_sample=False
    )
    spec_results.append({
        "prompt": prompt, "output": text, "new_tokens": n_tok,
        "seconds": round(elapsed, 2),
        "tok_per_sec": round(n_tok / elapsed, 2) if elapsed > 0 else None,
    })
    print(f"{prompt[:50]}... -> {n_tok} tok in {elapsed:.2f}s ({n_tok/elapsed:.1f} tok/s)")

spec_df = pd.DataFrame(spec_results)
spec_df.to_csv(ROOT / "part_c_speculative_results.csv", index=False)
spec_df


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Free VRAM before draft model load: 69.0 GB


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Explain how quantization reduces the computational... -> 150 tok in 6.90s (21.7 tok/s)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Compare the challenges of object detection in ther... -> 150 tok in 5.81s (25.8 tok/s)


,prompt,output,new_tokens,seconds,tok_per_sec
0,What is the main architectural difference betw...,How does this difference impact the performan...,150,5.90,25.42
1,Explain how quantization reduces the computati...,Quantization is a technique used to reduce th...,150,6.90,21.74
2,Compare the challenges of object detection in ...,Object detection in thermal infrared (TIR) im...,150,5.81,25.84


**Observations.** Speculative decoding was slower than plain greedy decoding on this run too: 21.7-25.8 tok/s versus 35.6-40.9 tok/s greedy for the same 3 prompts -- a 29-47% throughput **loss**, confirming the same finding as the RunPod run on a completely different GPU (A100 vs RTX 4090/5090) and a different draft model (Qwen2.5-0.5B base here vs Qwen2.5-1.5B-Instruct on RunPod). Even with an architecturally cleaner draft/target tokenizer pairing (both base-family Qwen2.5 checkpoints), passing explicit `tokenizer=`/`assistant_tokenizer=` arguments still forces the slower "universal assisted decoding" code path rather than the faster same-tokenizer fast path -- the cross-checkpoint verification overhead outweighs the benefit of skipping target-model forward passes for accepted draft tokens. This suggests the throughput loss is structural to universal assisted decoding in this transformers version, not specific to one GPU or draft-model choice.

### Step 3 — 4-bit Quantization & Production Cost [3 Marks]


In [17]:
# 10 distinct, genuinely in-domain prompts drawn from the held-out eval split (Part B1) -- more
# representative of production traffic than repeating the same 3 prompts.
DOMAIN_PROMPTS_10 = [r["instruction"] for r in eval_rows[:10]]
assert len(DOMAIN_PROMPTS_10) == 10, f"eval split only has {len(eval_rows)} rows, need >= 10"

BEST_STRATEGY_KWARGS = DECODING_CONFIGS["top_p"]  # replace with whichever strategy Step 1 recommends

def benchmark_config(model, tok, prompts, **gen_kwargs):
    torch.cuda.reset_peak_memory_stats()
    total_tokens, total_time = 0, 0.0
    for p in prompts:
        _, n_tok, elapsed = generate(model, tok, p, max_new_tokens=150, **gen_kwargs)
        total_tokens += n_tok
        total_time += elapsed
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9
    throughput = total_tokens / total_time if total_time > 0 else 0
    return peak_vram_gb, throughput

HOURLY_RATE_A100 = 12.00  # rupees/hour

def cost_per_million_tokens(throughput_tok_per_sec, hourly_rate):
    return (1_000_000 / throughput_tok_per_sec) / 3600 * hourly_rate

results_table = {}

# Reload base_model fresh again -- Step 2's speculative-decoding calls leave more KV-cache
# residue on it that plain empty_cache() won't reclaim (see Step 2 for why).
del base_model
gc.collect()
torch.cuda.empty_cache()
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map={"": 0}
)

# --- bfloat16 baseline (reuse base_model already loaded) ---
vram_bf16, tput_bf16 = benchmark_config(base_model, tokenizer, DOMAIN_PROMPTS_10, **BEST_STRATEGY_KWARGS)
results_table["bfloat16"] = (vram_bf16, tput_bf16, cost_per_million_tokens(tput_bf16, HOURLY_RATE_A100))

# Free the bf16 copy before loading the 4-bit copy -- keeps peak VRAM low enough to stay
# portable down to a 16GB T4, not just this session's GPU.
del base_model
gc.collect()
torch.cuda.empty_cache()

# --- 4-bit NF4 quantized ---
quant_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb_config, device_map={"": 0}
)
vram_4bit, tput_4bit = benchmark_config(quant_model, tokenizer, DOMAIN_PROMPTS_10, **BEST_STRATEGY_KWARGS)
results_table["4bit_nf4"] = (vram_4bit, tput_4bit, cost_per_million_tokens(tput_4bit, HOURLY_RATE_A100))

# --- 4-bit + speculative decoding ---
torch.cuda.reset_peak_memory_stats()
total_tokens, total_time = 0, 0.0
for p in DOMAIN_PROMPTS_10:
    _, n_tok, elapsed = generate_speculative(quant_model, draft_model, tokenizer, draft_tokenizer, p, max_new_tokens=150, do_sample=False)
    total_tokens += n_tok
    total_time += elapsed
vram_spec = torch.cuda.max_memory_allocated() / 1e9
tput_spec = total_tokens / total_time if total_time > 0 else 0
results_table["4bit_speculative"] = (vram_spec, tput_spec, cost_per_million_tokens(tput_spec, HOURLY_RATE_A100))

cost_df = pd.DataFrame(results_table, index=["peak_vram_gb", "throughput_tok_s", "cost_per_1M_tokens_inr"]).T
cost_df.to_csv(ROOT / "part_c_cost_table.csv")
cost_df


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`

,peak_vram_gb,throughput_tok_s,cost_per_1M_tokens_inr
bfloat16,16.316161,42.026456,79.315118
4bit_nf4,6.900025,19.131127,174.236122
4bit_speculative,6.942265,17.997622,185.209648


**Recommendation.** On this A100 80GB, bfloat16 is again the cheapest configuration at ₹79.32/1M tokens (16.3GB peak VRAM), versus ₹174.24 for 4-bit NF4 and ₹185.21 for 4-bit + speculative -- an even larger gap than observed on the RunPod run, since 4-bit throughput here (19.1 tok/s) is less than half of bfloat16's (42.0 tok/s). With 80GB of VRAM available, there is no memory pressure to justify 4-bit's dequantization overhead, so bfloat16 is the clear choice to deploy here; 4-bit NF4 remains the right choice only when VRAM, not cost, is the binding constraint (e.g. a 16GB T4 that cannot fit the full bfloat16 model at all).


## Optional Extension (No Marks)

Load the fine-tuned adapter (`./qlora_adapter_B`) on top of the 4-bit quantized base model and
re-run the Step 2 speculative decoding benchmark, to measure the realistic production
configuration (adapter + 4-bit + speculative decoding) against the base-model-only numbers above.


In [18]:
adapter_quant_model = PeftModel.from_pretrained(quant_model, "./qlora_adapter_B")
adapter_quant_model.eval()

ext_results = []
for prompt in DOMAIN_PROMPTS:
    text, n_tok, elapsed = generate_speculative(
    adapter_quant_model, draft_model, tokenizer, draft_tokenizer, prompt, max_new_tokens=150, do_sample=False
)
    ext_results.append({
        "prompt": prompt, "output": text, "new_tokens": n_tok,
        "tok_per_sec": round(n_tok / elapsed, 2) if elapsed > 0 else None,
    })
pd.DataFrame(ext_results)


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


,prompt,output,new_tokens,tok_per_sec
0,What is the main architectural difference betw...,How does this difference impact the performan...,150,19.29
1,Explain how quantization reduces the computati...,Quantization is a technique used to reduce th...,150,16.94
2,Compare the challenges of object detection in ...,Object detection in thermal infrared (IR) ima...,150,17.67
